# Check report by hand

In [ ]:
import subprocess
import sys
from pathlib import Path

import Scriptum
import yaml

# Derived from where this notebook sits, so moving the worktree does not
# turn the check below into a false alarm. Jupyter starts in the notebook's
# own directory, and this cell runs before anything changes it.
NOTEBOOK_DIR = Path.cwd()
WORKTREE = NOTEBOOK_DIR.parents[3]
here = Path(Scriptum.__file__).resolve().parent.parent

print('python    ', sys.executable)
print('Scriptum  ', Path(Scriptum.__file__).resolve())
print('PyYAML    ', yaml.__version__)
print('branch    ', subprocess.run(['git', 'branch', '--show-current'], cwd=here,
                                   capture_output=True, text=True).stdout.strip())

if here != WORKTREE:
    print()
    print('WARNING: Scriptum is NOT coming from this worktree.')
    print('         Expected', WORKTREE)
    print("         Pick this worktree's .venv kernel - see tests/Instructions.md.")
else:
    print()
    print('OK: importing from this worktree.')


## A workspace

The run writes a document and needs its data beside it, so everything is copied
to a temp directory. The repo stays clean, and you can throw the directory away.


In [ ]:
import os
import shutil
import tempfile

REPORT_DIR = WORKTREE / 'tests' / '02_basetest' / 'docx_basic' / 'text'

def workspace():
    """A fresh directory holding the fixtures, the template and the data."""
    work = Path(tempfile.mkdtemp(prefix='scriptum-'))
    for pattern in ('*.yaml', 'template.docx'):
        for path in REPORT_DIR.glob(pattern):
            shutil.copy(path, work)
    shutil.copytree(REPORT_DIR / 'data', work / 'data', dirs_exist_ok=True)
    os.chdir(work)
    return work

WORK = workspace()
print('working in', WORK)
print(sorted(p.name for p in WORK.iterdir()))


## 1. Read the document

`ReportDataFile` reads a `.yaml` document through `Scriptum.rdf.loader`;
anything else is refused with a message. The `.rdf` text parser is gone.

A broken document raises `DocumentError` carrying **every** diagnostic, not
just the first.


In [ ]:
rdf = Scriptum.ReportDataFile('word_text.yaml')

print('documenttype:', rdf.settings.documenttype)
print('datadir     :', rdf.settings.datadir)
print('tasks       :', len(rdf.tasks))
print('errors      :', rdf.errors or 'none')


## 2. What the tasks say

`what` is the operation, `where` the marker an *add* lands at, `target` the
**template name** — the tag written in the .docx — and the address the
**instance**: thus, for instance, `subsection:instruction::2` is the second copy of that block.

Note the `_global_` tasks at the end: global fills are applied last, and the
task list carries that rule so no back end has to remember it.


In [ ]:
def show_tasks(tasks, limit=None, only=None):
    rows = [t for t in tasks if only is None or only in '.'.join(t.myAddress)]
    print(f'{"#":>4}  {"what":6} {"where":18} {"target":22} address')
    print('-' * 110)
    for t in rows[:limit]:
        print(f'{t.serial:>4}  {t.what or "-":6} {t.where or "-":18} '
              f'{t.target or "-":22} {".".join(t.myAddress)}')
    if limit and len(rows) > limit:
        print(f'... {len(rows) - limit} more')

show_tasks(rdf.tasks, limit=40)


Try `show_tasks(rdf.tasks, only='...')` to see just the repeated blocks


In [ ]:
show_tasks(rdf.tasks, only='section:second')


## 3. Build the document

Anything the back end could not place prints a `WARNING`. A clean run prints
none.


In [ ]:
import contextlib
import io as _io

with contextlib.redirect_stdout(_io.StringIO()) as printed:
    managed = Scriptum.ManagedDocx('template.docx', rdf)
    managed.typesetting(rdf)
    managed.save('result.docx',finish=False)

complaints = [line for line in printed.getvalue().splitlines()
              if 'WARNING' in line or 'ERROR' in line or 'INFO' in line]
print('written:', WORK / 'result.docx')
print('complaints:', len(complaints))
for line in complaints[:20]:
    print('  ', line)


## 4. Read it back

Open created `*.docx` in Word if you want to look at the formatting; this shows
what it *says*, which is what the automated comparison uses.


In [ ]:
import docx

def spoken(path):
    document = docx.Document(path)
    said = [p.text.strip() for p in document.paragraphs]
    for table in document.tables:
        for row in table.rows:
            said.extend(cell.text.strip() for cell in row.cells)
    return [line for line in said if line]

lines = spoken(WORK / 'result.docx')
print(len(lines), 'non-empty lines')
for line in lines[:30]:
    print('  ', line[:100])


## 5. Compare with the reference

`expected/*.json`, beside this notebook, is what this
fixture's **`.rdf`** produced before the back end changed (the `.rdf` and its
parser are gone; the reference is their record). Digits and weekday
names are collapsed on both sides, because the reference was captured on
another day and `date:now` is evaluated per run.

An empty report below means the YAML document says exactly what the text one
said.


In [ ]:
import json
import re

DIGITS = re.compile(r'\d+')
WEEKDAY = re.compile(r'\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b')

def normalise(lines):
    return [WEEKDAY.sub('#', DIGITS.sub('#', line)) for line in lines]

REFERENCE = REPORT_DIR / 'expected' / 'word_text.json'

expected = normalise(json.loads(REFERENCE.read_text(encoding='utf-8')))
got = normalise(spoken(WORK / 'result.docx'))

print(f'reference {len(expected)} lines, this run {len(got)} lines')
if expected == got:
    print('IDENTICAL')
else:
    for i, (a, b) in enumerate(zip(expected, got)):
        if a != b:
            print(f'first difference at line {i}')
            print('  reference:', a[:110])
            print('  this run :', b[:110])
            break
    else:
        print('one is a prefix of the other')
